<a href="https://colab.research.google.com/github/Jaswanth431/DL-Assignment-1/blob/main/DL_Assignment_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
# @title
!pip install wandb

In [24]:
#importing packages
from keras.datasets import fashion_mnist
import pandas as pd
import numpy as np
import wandb
import math


In [25]:
#creating wandb connection
try:
    wandb.login(key="62cfafb7157dfba7fdd6132ac9d757ccd913aaaf")
    wandb.init(project="DL assignment 1")
    print("Wandb connection initiated")
except:
    print("error in wandb")

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


Wandb connection initiated


In [26]:
#getting training and test data
[(x_total_train_data, y_total_train_data), (x_test_data, y_test_data)] = fashion_mnist.load_data()
total_train_len = len(x_total_train_data);
train_count = int(total_train_len*.90)
flattened_train_data = []
#flattening the 28*28 pixel matrix
for i in range(0, total_train_len):
    flattened_train_data.append(x_total_train_data[i].flatten())

x_train_data = np.array(flattened_train_data[0:train_count])
y_train_data = y_total_train_data[0:train_count]
x_validation_data = np.array(flattened_train_data[train_count:])
y_validation_data = y_total_train_data[train_count:]
# print(x_train_data[0])

In [28]:
#Creating neural network
class NeuralNetwork:
    def __init__(self, hidden_layers, hidden_layer_neurons, input_layer_neurons, output_layer_neurons  ):
        #initializing values
        self.hidden_layers = hidden_layers
        self.hidden_layer_neurons = hidden_layer_neurons
        self.input_layer_neurons = input_layer_neurons
        self.output_layer_neurons = output_layer_neurons
        self.total_layers = self.hidden_layers+1
        self.output_layer_number = self.total_layers - 1;

        #input weight and bias initilization
        self.w = []
        self.b = []

        #weights and bias initialization for hidden layers
        for i in range(0, self.total_layers):
          if i == 0:
            temp1 = np.random.rand(self.hidden_layer_neurons, self.input_layer_neurons)
            temp2 = np.zeros(self.hidden_layer_neurons)
          elif i==self.total_layers -1:
            temp1 = np.random.rand(self.output_layer_neurons, self.hidden_layer_neurons)
            temp2 = np.zeros(self.output_layer_neurons)
          else:
            temp1  =  np.random.rand( self.hidden_layer_neurons, self.hidden_layer_neurons)
            temp2 = np.zeros(self.hidden_layer_neurons)

          self.w.append(temp1)
          self.b.append(temp2)
        # print(self.w, self.b)
        # for i in range(0, self.total_layers):
        #    print(self.b[i].shape)
        # print(self.w[2].shape)


    def forward_propogate(self, input):
        h = [None] * self.total_layers
        a = [None] * self.total_layers

        for i in range(0, self.total_layers):
            if(i == 0):
              a[i] = np.dot(self.w[i],input ) + self.b[i]
              h[i] = self.sigmoid(a[i])
            elif i == self.total_layers-1:
              a[i] = np.dot(self.w[i],h[i-1] ) + self.b[i]
              h[i] = self.softmax(a[i])
            else:
              a[i] = np.dot(self.w[i],h[i-1] ) + self.b[i]
              h[i] = self.sigmoid(a[i])
        return h, a



    def back_propagation(self, h, a, actual_class, input_pixels):
       d_h = [None] * self.total_layers
       d_a =  [None] * self.total_layers
       d_w =  [None] * self.total_layers
       d_b =  [None] * self.total_layers
       y_original = np.zeros(self.output_layer_neurons)
       y_original[actual_class] = 1

       #gradient w.r.p.t output y hat
       d_a[self.total_layers-1] = -(y_original - h[self.total_layers-1])
      #  print(d_a[self.total_layers-1].shape)
      #  print(d_a[self.total_layers-1])

       for i in range(self.total_layers-1, -1, -1):
        if(i == 0):
          d_w[i] = np.dot(d_a[i].reshape(-1, 1), input_pixels.reshape(1, -1))
        else:
          d_w[i] = np.dot(d_a[i].reshape(-1, 1), h[i-1].reshape(1, -1))

        d_b[i] = d_a[i]
        if(i-1>=0):
          d_h[i-1]=np.dot(self.w[i].T,d_a[i])
          d_a[i-1] = d_h[i-1] * self.sigmoid_derivative(a[i-1])
       return d_w, d_b

    def gradient_descent(self, x_train_data, y_train_data):
      max_iterations = 10
      for i in range(0, max_iterations):
        d_w = [np.zeros_like(weight) for weight in self.w]
        d_b = [np.zeros_like(bias) for bias in self.b]
        for j in range(0, len(x_train_data)):
          h,a = self.forward_propogate(x_train_data[j])
          d_w_temp, d_b_temp = self.back_propagation(h, a, y_train_data[j], x_train_data[j])
          for k in range(self.total_layers):
                d_w[k] += d_w_temp[k]
                d_b[k] += d_b_temp[k]
        self.update_parameters(d_w, d_b, 0.0001)

    def calculate_train_loss(self, x_train_data, y_train_data):
      count = 0;
      for i in range(len(x_train_data)):
        h,a = self.forward_propogate(x_train_data[i])
        output_class = np.argmax(h[self.total_layers-1])
        if(output_class == y_train_data[i]):
          count+=1
      print(count)
      print(count/len(x_train_data))


    def update_parameters(self,d_w,d_b, eta):
      for i in range(0, self.total_layers):
        self.w[i] -= eta*d_w[i]
        self.b[i] -= eta*d_b[i]

    def sigmoid(self,arr):
        return 1. / (1.+np.exp(-arr))
    def sigmoid_derivative(self, arr):
        return self.sigmoid(arr) * (1-self.sigmoid(arr))
    def softmax(self, arr):
        return np.exp(arr) / np.sum(np.exp(arr), axis=0)



n_network = NeuralNetwork(3, 20, 784, 10)
n_network.gradient_descent(x_train_data, y_train_data)
# n_network.calculate_train_loss(x_train_data, y_train_data)
# h, a = n_network.forward_propogate(x_train_data[0])
# d_w, d_b = n_network.back_propagation(h, a, y_train_data[0], x_train_data[0])

# print(y_train_data[0])



In [29]:
n_network.calculate_train_loss(x_train_data, y_train_data)


5409
0.10016666666666667
